# Tahap 3 Nasional — Tarik Data Cuaca Riil dari NASA POWER

Versi nasional dari notebook cuaca regional. Menarik data cuaca bulanan riil
(2013-2023, 11 tahun) untuk 3 kategori nasional (Hydro, Solar, Wind) dari
NASA POWER API.

**Koordinat (hasil Tahap 2 Nasional):**
- **Hydro** — centroid tertimbang 3 klaster PLTA besar (Jawa Barat/Tengah,
  Sulawesi Tengah/Poso, Sumatera Utara) — bukan registry lengkap, simplifikasi
  dari pembangkit terbesar yang terdokumentasi publik
- **Solar** — titik representatif Jawa Barat (area Cirata), simplifikasi
- **Wind** — SAMA PERSIS dengan koordinat regional Sulsel — karena PLTB
  Sidrap + Tolo/Jeneponto mencakup ~97% kapasitas Wind nasional

Jalankan sel-sel di bawah secara berurutan. Butuh koneksi internet aktif.

In [ ]:
!pip install requests pandas -q

## 1. Konfigurasi

In [ ]:
import requests
import pandas as pd

# Konfigurasi 3 titik koordinat kategori NASIONAL (hasil Tahap 2 Nasional)
LOKASI = {
    "Hydro": {"lat": -4.40, "lon": 107.95, "parameter": "PRECTOTCORR"},
    "Solar": {"lat": -6.85, "lon": 107.40, "parameter": "ALLSKY_SFC_SW_DWN"},
    "Wind":  {"lat": -4.64, "lon": 119.82, "parameter": "WS10M"},  # sama dengan regional
}

START_YEAR = 2013
END_YEAR = 2023  # 11 tahun, sesuai cakupan HEESI Tahap 1 Nasional

BASE_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"

print("Konfigurasi siap:")
for kategori, info in LOKASI.items():
    print(f"  {kategori}: lat={info['lat']}, lon={info['lon']}, parameter={info['parameter']}")
print(f"\nRentang tahun: {START_YEAR}-{END_YEAR}")

## 2. Fungsi penarik data

In [ ]:
def tarik_data(nama_kategori: str, lat: float, lon: float, parameter: str) -> list:
    """Tarik satu parameter cuaca bulanan untuk satu titik koordinat."""
    params = {
        "parameters": parameter,
        "community": "RE",  # Renewable Energy community
        "longitude": lon,
        "latitude": lat,
        "format": "JSON",
        "start": START_YEAR,
        "end": END_YEAR,
    }

    print(f"  Menarik {nama_kategori} ({parameter}) di ({lat}, {lon})...")
    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    try:
        monthly = data["properties"]["parameter"][parameter]
    except KeyError:
        raise RuntimeError(
            f"Struktur respons tidak sesuai ekspektasi untuk {nama_kategori}. "
            f"Cek manual: {resp.url}"
        )

    rows = []
    for key, value in monthly.items():
        # Format key: "YYYYMM", contoh "201301".
        # Bulan "13" = rata-rata tahunan, bukan bulan sungguhan -> dilewati.
        tahun, bulan = key[:4], key[4:6]
        if bulan == "13":
            continue
        if value == -999.0:  # kode missing value standar NASA POWER
            print(f"    [PERINGATAN] Data hilang untuk {tahun}-{bulan}, dilewati")
            continue
        rows.append({
            "Tahun": int(tahun),
            "Bulan": int(bulan),
            "Kategori": nama_kategori,
            "Parameter_Cuaca": parameter,
            "Nilai_Cuaca": value,
            "Lat": lat,
            "Lon": lon,
        })
    return rows

## 3. Eksekusi — tarik data untuk 3 kategori, 11 tahun

In [ ]:
semua_data = []
print(f"Menarik data cuaca NASA POWER, {START_YEAR}-{END_YEAR}...\n")

for kategori, info in LOKASI.items():
    hasil = tarik_data(kategori, info["lat"], info["lon"], info["parameter"])
    semua_data.extend(hasil)
    print(f"    -> {len(hasil)} baris berhasil ditarik\n")

df = pd.DataFrame(semua_data)
df = df.sort_values(["Kategori", "Tahun", "Bulan"]).reset_index(drop=True)
print(f"Total baris ditarik: {len(df)}")

## 4. Validasi

Harus 132 baris per kategori (12 bulan x 11 tahun 2013-2023) kalau semua berhasil.

In [ ]:
print("Jumlah baris per kategori:")
print(df.groupby("Kategori").size())

print("\nCek missing value per kategori:")
for kategori in LOKASI:
    jumlah = len(df[df["Kategori"] == kategori])
    status = "OK" if jumlah == 132 else f"KURANG ({jumlah}/132) - cek peringatan di atas"
    print(f"  {kategori}: {status}")

## 5. Pratinjau & simpan ke CSV

In [ ]:
df.head(15)

In [ ]:
out_path = "cuaca_riil_nasional.csv"
df.to_csv(out_path, index=False)
print(f"Disimpan ke {out_path}")

# Kalau di Google Colab, unduh otomatis:
try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    print("(Bukan lingkungan Colab -- file tersimpan di direktori kerja lokal)")

## Langkah berikutnya

Setelah `cuaca_riil_nasional.csv` berhasil dibuat, lanjut ke **Tahap 4 Nasional**:
disagregasi tahunan -> bulanan memakai `disagregasi_regional.py` yang sudah ada
(cukup ganti dictionary `ANNUAL` ke 11 tahun data nasional dan path file cuaca).